In [2]:
import pickle
from rdkit import Chem
import torch
from torch_geometric.data import InMemoryDataset
import pandas as pd
import os


from reduceGraph import mol_to_graph, graph_to_pyg, reduce_graph_from_mol, mol_to_pool_idx, graph_to_pyg_oh, reduce_graph_from_mol_oh

In [3]:
#Load dataset 

with open('datasets/df_random_vs_st_dataset_revised.pkl', 'rb') as f:
    data = pickle.load(f)

In [43]:
data

,nonstereo_aromatic_smiles,accession,chembl_cid,chembl_tid,label
0,Brc1ccc(-c2cnc3nnc(Cc4c[nH]c5ccccc45)n3n2)cc1,P08581,CHEMBL3799231,CHEMBL3717,1
1,Brc1ccc(-c2cnc3nnc(Cc4ccc5ncccc5c4)n3n2)cc1,P08581,CHEMBL3798130,CHEMBL3717,1
2,Brc1ccc(-c2nc3ccc(Nc4ncnc5ccccc45)cc3[nH]2)cc1,P08581,CHEMBL3321905,CHEMBL3717,1
3,C#CCC(Oc1c(N)ncc2c(-c3cnn(C4CCNCC4)c3)coc12)c1...,P08581,CHEMBL2401818,CHEMBL3717,1
4,C#Cc1c(Oc2ccc(-n3cc(C(=O)NCc4ccc(F)cc4)ccc3=O)...,P08581,CHEMBL3309958,CHEMBL3717,1
...,...,...,...,...,...
43069,CN(CCC(=O)N1CCN(c2ccc(F)c(F)c2)CC1)CCC1c2ccccc...,P27487,CHEMBL473763,CHEMBL1917,0
43070,COc1ccc(COc2ccc(C3CN(C(=O)c4cc(OCCO)ccn4)C3)cc...,P27487,CHEMBL4543929,CHEMBL1844,0
43071,NC(CCC(=O)NC(CSC(=O)N(O)c1ccc(Cl)cc1)C(=O)NCC(...,P27487,CHEMBL131578,CHEMBL2424,0
43072,CC1CCc2ccccc2N1C(=O)c1ccc2c(c1)NC(=O)CO2,P27487,CHEMBL4474881,CHEMBL1994,0


In [44]:
#list of protein targets in dataset
targets = data['accession'].unique()
targets

array(['P08581', 'P35968', 'Q16790', 'P56817', 'P22303', 'P06276',
       'P00915', 'P34913', 'Q13547', 'P27487'], dtype=object)

In [45]:

def smiles_to_data(smiles, label):
    """
    Converts smiles to pytorch geometric graph object. takes smiles as string and label as binary (active/inactive). 
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    graph = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(graph)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    """
    turns dataframe into list of pytorch geometric objects

    parameters: 
    df : dataframe 
    smiles_col (string) : name of column containing smiles
    label_col (string) : name of column containing labels
    """
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [46]:
#Create pythorch geometric datasets for each protein target 

#  Whole graph data (for GAT model)

for target in targets:
    filtered_data = data[data['accession'] == target]
    dataset = dataframe_to_pyg_dataset(filtered_data)
    os.makedirs('Gdatasets', exist_ok=True)
    torch.save(dataset, f'Gdatasets/{target}_dataset.pt')
    dataset = torch.load(f'Gdatasets/{target}_dataset.pt', weights_only=False)
    print(target, len(dataset))

P08581 2828
P35968 5046
Q16790 7558
P56817 2760
P22303 4136
P06276 2610
P00915 7716
P34913 2884
Q13547 4056
P27487 3480


In [47]:
#  reduced graph data (for GAT-rg model)

def smiles_to_rgdata(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    data = reduce_graph_from_mol_oh(mol)  # pytorch geometric RG graph from mol 
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_rg_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgdata(smiles, label)
        if data is not None:
            #add smiles to dataset 
            data.smiles = smiles            
            data_list.append(data)

    return data_list

In [48]:
#  reduced graph data (for GAT-rg model)
for target in targets:
    filtered_data = data[data['accession'] == target]
    dataset = dataframe_to_rg_pyg_dataset(filtered_data)
    os.makedirs('RGdatasets', exist_ok=True)
    torch.save(dataset, f'RGdatasets/{target}_RG_dataset.pt')
    dataset = torch.load(f'RGdatasets/{target}_RG_dataset.pt', weights_only=False)
    print(target, 'RG', len(dataset))


P08581 RG 2828
P35968 RG 5046
Q16790 RG 7558
P56817 RG 2760
P22303 RG 4136
P06276 RG 2610
P00915 RG 7716
P34913 RG 2884
Q13547 RG 4056
P27487 RG 3480


In [49]:
#  pooling graph data (whole graph with pooling tensors (for PP-GAT model)



def smiles_to_rgnn_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None  
    G = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(G)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 

    pharma_index, new_edge_index, new_edge_attr = mol_to_pool_idx(mol)
    data.pharma_index = pharma_index
    data.new_edge_index = new_edge_index
    data.new_edge_attr = new_edge_attr

    return data

def dataframe_to_rgnn_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgnn_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list


In [50]:
for target in targets:
    filtered_data = data[data['accession'] == target]
    dataset = dataframe_to_rgnn_pyg_dataset(filtered_data)
    os.makedirs('PPdatasets', exist_ok=True)
    torch.save(dataset, f'PPdatasets/{target}_RG_dataset.pt')
    dataset = torch.load(f'PPdatasets/{target}_RG_dataset.pt', weights_only=False)
    print(target, 'PPGAT', len(dataset))

P08581 PPGAT 2828
P35968 PPGAT 5046
Q16790 PPGAT 7558
P56817 PPGAT 2760
P22303 PPGAT 4136
P06276 PPGAT 2610
P00915 PPGAT 7716
P34913 PPGAT 2884
Q13547 PPGAT 4056
P27487 PPGAT 3480
